<a href="https://colab.research.google.com/github/Nourin-Nusrat/CSE4261_DNN/blob/main/Assignment1/DNNAssignment1_resnet152v2_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import warnings
import sys
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

In [ ]:
import keras
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from keras import layers
import numpy as np

In [ ]:
(x_train_tmp, y_train_tmp), (x_test_tmp, y_test_tmp) = keras.datasets.cifar100.load_data()

train_filter = (y_train_tmp < 20).flatten()
test_filter = (y_test_tmp < 20).flatten()

x_train = x_train_tmp[train_filter]
y_train = y_train_tmp[train_filter]
x_test = x_test_tmp[test_filter]
y_test = y_test_tmp[test_filter]

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

trainY = to_categorical(y_train, num_classes=20)
testY = to_categorical(y_test, num_classes=20)

169001437/169001437 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [ ]:
input_shape = (32, 32, 3)
resnet152v2_model = keras.applications.ResNet152V2(
    include_top=False,
    weights="imagenet",
    input_shape=input_shape,
    pooling='avg'
)
model = keras.Sequential(
    [
        keras.Input(shape=(32, 32, 3)),
        resnet152v2_model,

        layers.Flatten(),

        layers.Dropout(0.4),
        layers.Dense(512),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Dropout(0.4),
        layers.Dense(127),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Dense(20, activation='softmax')
    ]
)
model.summary()

234545216/234545216 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet152v2 (Functional)        │ (None, 2048)           │    58,331,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 127)            │        65,151 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 127)            │           508 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 127)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 20)             │         2,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 59,451,003 (226.79 MB)

 Trainable params: 59,305,981 (226.23 MB)

 Non-trainable params: 145,022 (566.49 KB)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)

epochs = 10
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss=keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=['accuracy',],
)

model.fit(x_train, trainY, epochs=epochs, callbacks=[early_stopping], validation_split=0.1)
model.save('resnet152v2_model.keras')

Epoch 1/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 226s 303ms/step - accuracy: 0.0809 - loss: 3.0759 - val_accuracy: 0.0930 - val_loss: 4.3651
Epoch 2/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 30s 89ms/step - accuracy: 0.1343 - loss: 2.8380 - val_accuracy: 0.0700 - val_loss: 2.9889
Epoch 3/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 40s 87ms/step - accuracy: 0.1717 - loss: 2.7185 - val_accuracy: 0.2290 - val_loss: 3.0966
Epoch 4/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 41s 88ms/step - accuracy: 0.2247 - loss: 2.5221 - val_accuracy: 0.2200 - val_loss: 2.7491
Epoch 5/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 41s 88ms/step - accuracy: 0.2254 - loss: 2.4881 - val_accuracy: 0.2850 - val_loss: 2.3707
Epoch 6/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 41s 87ms/step - accuracy: 0.2694 - loss: 2.3498 - val_accuracy: 0.1640 - val_loss: 3.9530
Epoch 7/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 41s 86ms/step - accuracy: 0.2267 - loss: 2.4733 - val_accuracy: 0.2290 - val_loss: 2.6451
Epoch 8/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 41s 87ms/step - accuracy: 0.2286 - loss: 2.4840 

In [ ]:
model.evaluate(x_test, testY)

63/63 ━━━━━━━━━━━━━━━━━━━━ 7s 105ms/step - accuracy: 0.2645 - loss: 2.3610


[2.3306872844696045, 0.27250000834465027]